# 1. Imports 

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import optuna
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

# 2. Configuration

In [ ]:
sys.path.append("../src")
from preprocessing import (
    get_dataloaders, get_device,
    WasteDataset, CLASSES, NUM_CLASSES, IDX_TO_CLASS,
)

device = get_device()
print(f"Device  : {device}")
print(f"Classes : {NUM_CLASSES} — {CLASSES}")

# 3. MLflow + load data

In [ ]:
MLFLOW_URI       = "http://localhost:5000"
EXPERIMENT_NAME  = "sortify-mobilenet"
REGISTERED_MODEL = "sortify-cnn-mobilenet"
SPLITS_PATH      = "../data/splits/splits.json"
SAVED_DIR        = Path("../saved")
SAVED_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

class_weights = WasteDataset(splits["train"]).get_class_weights().to(device)

print(f"Train : {len(splits['train'])} | Val : {len(splits['val'])} | Test : {len(splits['test'])}")
print(f"Class weights : {[round(w, 3) for w in class_weights.tolist()]}")

# 4. MobileNetV2 CNN Architecture
Only the final classifier is replaced. The ImageNet weights are retained in the feature extractor.

Two fine-tuning strategies:
- **frozen** : We only train the classifier (fast, requires less data)
- **unfrozen** : We enable the last convolutional layers (higher accuracy, longer processing time)

In [ ]:
class MobileNetWaste(nn.Module):
    """
    MobileNetV2 pretrained on ImageNet with a custom classifier.
    Input  : (B, 3, 224, 224)
    Output : (B, NUM_CLASSES)
    """

    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = 0.3, freeze_backbone: bool = True) -> None:
        super().__init__()
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

        if freeze_backbone:
            for param in backbone.features.parameters():
                param.requires_grad = False
        else:
            # Debloque uniquement les 4 derniers blocs
            for param in backbone.features[:-4].parameters():
                param.requires_grad = False

        self.features   = backbone.features
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.pool(self.features(x)))


# Verification
_m   = MobileNetWaste().to(device)
_out = _m(torch.randn(2, 3, 224, 224).to(device))
total     = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Output shape       : {_out.shape}")
print(f"Params totaux      : {total:,}")
print(f"Params trainables  : {trainable:,} ({trainable/total*100:.1f}%)")
del _m, _out

# 5. Optuna — Hyperparameter tuning
Each trial = 1 MLflow run. We are looking for the learning rate, dropout, batch size, backbone freeze, and optimizer.

In [ ]:
N_TRIALS      = 20
EPOCHS_OPTUNA = 8   # shorter than scratch because the pretrained model converges quickly


def objective(trial: optuna.Trial) -> float:
    lr              = trial.suggest_float("lr",           1e-5, 1e-3, log=True)
    dropout         = trial.suggest_float("dropout",      0.1, 0.5)
    batch_size      = trial.suggest_categorical("batch_size", [16, 32, 64])
    weight_decay    = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    freeze_backbone = trial.suggest_categorical("freeze_backbone", [True, False])
    optimizer_name  = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])

    # pretrained=True -> ImageNet normalization
    train_loader, val_loader, _ = get_dataloaders(
        SPLITS_PATH, batch_size=batch_size, pretrained=True
    )
    model     = MobileNetWaste(dropout=dropout, freeze_backbone=freeze_backbone).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    opt       = (
        optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
        if optimizer_name == "Adam" else
        optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    )
    scheduler     = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best_val_loss = float("inf")
    no_improve    = 0

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.log_params({
            "lr": lr, "dropout": dropout, "batch_size": batch_size,
            "weight_decay": weight_decay, "optimizer": optimizer_name,
            "freeze_backbone": freeze_backbone, "phase": "optuna",
        })

        for epoch in range(1, EPOCHS_OPTUNA + 1):
            # ── train
            model.train()
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                opt.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                opt.step()

            # ── val
            model.eval()
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs  = model(images)
                    val_loss += criterion(outputs, labels).item() * images.size(0)
                    correct  += (outputs.argmax(1) == labels).sum().item()
                    total    += images.size(0)
            val_loss /= total
            val_acc   = correct / total

            scheduler.step(val_loss)
            mlflow.log_metrics({"val_loss": val_loss, "val_acc": val_acc}, step=epoch)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                no_improve    = 0
            else:
                no_improve += 1

            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if no_improve >= 3:
                break

        mlflow.log_metric("best_val_loss", best_val_loss)

    return best_val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    study_name="sortify-mobilenet-optuna",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print("\nMeilleurs hyperparametres :")
for k, v in best.items():
    print(f"  {k:<18} : {v}")
print(f"Meilleur val_loss : {study.best_value:.4f}")

In [ ]:
# Optuna Visualization 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_values = [t.value for t in study.trials if t.value is not None]
axes[0].plot(trial_values, marker="o", color="#7F77DD", markersize=5, linewidth=1)
axes[0].axhline(study.best_value, color="#D85A30", linestyle="--", label=f"Best: {study.best_value:.4f}")
axes[0].set_title("Val loss par trial")
axes[0].set_xlabel("Trial")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()), color="#5DCAA5")
axes[1].set_title("Importance des hyperparametres")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(SAVED_DIR / "mobilenet_optuna.png", dpi=150)
plt.show()

# 6. Final training with the best hyperparameters
Two phases:
- **Phase 1** : backbone frozen
- **Phase 2** : unfreeze the last 4 blocks — deep fine-tuning with reduced lr

In [ ]:
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 20
PATIENCE      = 5

train_loader, val_loader, test_loader = get_dataloaders(
    SPLITS_PATH, batch_size=best["batch_size"], pretrained=True
)
model     = MobileNetWaste(dropout=best["dropout"], freeze_backbone=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

history       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "phase": []}
best_val_loss = float("inf")
no_improve    = 0

with mlflow.start_run(run_name="final_training") as final_run:
    mlflow.log_params({
        **best,
        "epochs_phase1": EPOCHS_PHASE1,
        "epochs_phase2": EPOCHS_PHASE2,
        "architecture": "MobileNetV2",
        "phase": "final",
    })

    # ════════════════════════════════════
    # Phase 1 — backbone frozen
    # ════════════════════════════════════
    optimizer = (
        optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=best["lr"], weight_decay=best["weight_decay"])
        if best["optimizer"] == "Adam" else
        optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=best["lr"], weight_decay=best["weight_decay"])
    )
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-7)

    print("Phase 1 — backbone frozen")
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>9}")
    print("-" * 55)

    for epoch in range(1, EPOCHS_PHASE1 + 1):
        # train
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += images.size(0)
        train_loss, train_acc = t_loss / total, correct / total

        # val
        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs  = model(images)
                v_loss  += criterion(outputs, labels).item() * images.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total   += images.size(0)
        val_loss, val_acc = v_loss / total, correct / total

        scheduler.step(val_loss)
        for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc"], [train_loss, val_loss, train_acc, val_acc]):
            history[k].append(v)
        history["phase"].append(1)
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                            "train_acc": train_acc, "val_acc": val_acc}, step=epoch)

        print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} {train_acc:>10.4f} {val_acc:>9.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_loss": val_loss, "val_acc": val_acc,
                        "classes": CLASSES, "hyperparams": best},
                       SAVED_DIR / "mobilenet_best.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"Early stopping phase 1 a l'epoch {epoch}")
                break

    # ════════════════════════════════════
    # Phase 2 — unfreeze the last 4 blocks
    # ════════════════════════════════════
    for param in model.features[:-4].parameters():
        param.requires_grad = False
    for param in model.features[-4:].parameters():
        param.requires_grad = True

    # Learning rate reduced by a factor of 10 to avoid destroying the pretrained weights
    optimizer = (
        optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=best["lr"] / 10, weight_decay=best["weight_decay"])
        if best["optimizer"] == "Adam" else
        optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=best["lr"] / 10, weight_decay=best["weight_decay"])
    )
    scheduler  = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-8)
    no_improve = 0
    p1_epochs  = len(history["train_loss"])

    print("\nPhase 2 — unfreeze 4 derniers blocs (lr / 10)")
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>9}")
    print("-" * 55)

    for epoch in range(1, EPOCHS_PHASE2 + 1):
        # train
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += images.size(0)
        train_loss, train_acc = t_loss / total, correct / total

        # val
        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs  = model(images)
                v_loss  += criterion(outputs, labels).item() * images.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total   += images.size(0)
        val_loss, val_acc = v_loss / total, correct / total

        scheduler.step(val_loss)
        global_epoch = p1_epochs + epoch
        for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc"], [train_loss, val_loss, train_acc, val_acc]):
            history[k].append(v)
        history["phase"].append(2)
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                            "train_acc": train_acc, "val_acc": val_acc}, step=global_epoch)

        print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} {train_acc:>10.4f} {val_acc:>9.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save({"epoch": global_epoch, "model_state": model.state_dict(),
                        "val_loss": val_loss, "val_acc": val_acc,
                        "classes": CLASSES, "hyperparams": best},
                       SAVED_DIR / "mobilenet_best.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"Early stopping phase 2 a l'epoch {epoch}")
                break

    # ── test set
    ckpt = torch.load(SAVED_DIR / "mobilenet_best.pt", map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_preds.extend(model(images.to(device)).argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)

    test_acc = float((all_preds == all_labels).mean())
    test_f1  = float(f1_score(all_labels, all_preds, average="macro"))

    mlflow.log_metrics({"test_acc": test_acc, "test_f1_macro": test_f1, "best_val_loss": best_val_loss})
    mlflow.pytorch.log_model(model, artifact_path="model", registered_model_name=REGISTERED_MODEL)

    print(f"\nTest accuracy  : {test_acc:.4f}")
    print(f"Test F1 macro  : {test_f1:.4f}")
    print(f"Run ID         : {final_run.info.run_id}")

# 7. Learning curves with phase separation

In [ ]:
x = range(1, len(history["train_loss"]) + 1)
phase_split = history["phase"].index(2)  # First Epoch, Phase 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    ax.plot(x, history[f"train_{metric}"], label="train", color="#5DCAA5")
    ax.plot(x, history[f"val_{metric}"],   label="val",   color="#D85A30")
    ax.axvline(phase_split, color="#7F77DD", linestyle="--", alpha=0.7, label="unfreeze")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("MobileNetV2 — Courbes d'apprentissage (phase 1 | phase 2)", fontsize=13)
plt.tight_layout()
plt.savefig(SAVED_DIR / "mobilenet_learning_curves.png", dpi=150)
plt.show()

# 8. Confusion matrix + classification report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))

cm_norm = confusion_matrix(all_labels, all_preds).astype(float)
cm_norm /= cm_norm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=CLASSES, yticklabels=CLASSES,
            cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_xlabel("Predit", fontsize=12)
ax.set_ylabel("Reel", fontsize=12)
ax.set_title("Matrice de confusion normalisee — MobileNetV2", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(SAVED_DIR / "mobilenet_confusion_matrix.png", dpi=150)
plt.show()

# 10. Comparison with CNN Scratch + Model Registry promotion

In [ ]:
client   = MlflowClient(MLFLOW_URI)
versions = client.get_latest_versions(REGISTERED_MODEL)
last_ver = versions[-1].version

# Load the CNN scratch stats to compare
scratch_stats_path = SAVED_DIR / "scratch_stats.json"
if scratch_stats_path.exists():
    with open(scratch_stats_path) as f:
        scratch = json.load(f)
    print("Comparaison CNN Scratch vs MobileNetV2 :")
    print(f"  {'Modele':<20} {'Test Acc':>10} {'Test F1':>10}")
    print(f"  {'-'*42}")
    print(f"  {'CNN Scratch':<20} {scratch['test_accuracy']:>10.4f} {scratch['test_f1_macro']:>10.4f}")
    print(f"  {'MobileNetV2':<20} {test_acc:>10.4f} {test_f1:>10.4f}")
    gain = (test_acc - scratch["test_accuracy"]) * 100
    print(f"\n  Gain MobileNet : {gain:+.2f}%")
else:
    print(f"Test accuracy  : {test_acc:.4f}")
    print(f"Test F1 macro  : {test_f1:.4f}")

# Save the stats for comparison with YOLO
with open(SAVED_DIR / "mobilenet_stats.json", "w") as f:
    json.dump({"model": "mobilenet_v2", "test_accuracy": round(test_acc, 4),
               "test_f1_macro": round(test_f1, 4), "best_val_loss": round(best_val_loss, 4),
               "hyperparams": best}, f, indent=2)

# Promote to Staging if > 0.80 (threshold higher than scratch because it's pretrained)
if test_acc >= 0.80:
    client.transition_model_version_stage(
        name=REGISTERED_MODEL, version=last_ver, stage="Staging"
    )
    print(f"\nModele v{last_ver} promu en Staging (test_acc={test_acc:.4f})")
else:
    print(f"\nModele v{last_ver} reste en None — test_acc={test_acc:.4f} < 0.80")